In [ ]:
# IPython magic commands
%load_ext autoreload
%autoreload 2

# Standard library imports
import base64
import io
import os
from pathlib import Path

# Third-party imports
import ipywidgets as widgets
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import multivariate_normal
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# AEON imports
from aeon.schema.schemas import social02
from swc.aeon.io import api as aeon_api

# Custom utilities imports
from data_io_utils import load_data_from_parquet, save_all_experiment_data

# Definitions

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
save_dir = Path("/ceph/aeon/aeon/code/scratchpad/anaya/rl_modelling_results")
os.makedirs(data_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas
light_off, light_on = 7, 20  # 7am to 7pm
fps = 50

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]
experiment = experiments[0]

In [ ]:
def load_experiment_data(
    data_dir: str,
    experiment: dict | None = None,
    periods: list | None = None,
    data_types: list[str] = ['rfid', 'position'],
    trim_days: int | None = None
) -> dict:
    """
    Load all data types for specified periods of an experiment.
    
    Parameters:
    - experiment: experiment dict with period start/end times
    - periods: list of periods to load
    - data_types: list of data types to load
    - data_dir: directory containing data files
    - trim_days: Optional number of days to trim from start (None = no trim)
    
    Returns:
    - Dictionary containing dataframes for each period/data type combination
    """
    
    result = {}

    if periods is None:
        periods = [None]
    
    for period in periods:
        for data_type in data_types:
            print(f"Loading {period} {data_type} data...")
            
            # Load data
            if experiment is not None:
                experiment_name = experiment["name"]
            else:
                experiment_name = None
            df = load_data_from_parquet(
                experiment_name=experiment_name,
                period=period,
                data_type=data_type,
                data_dir=data_dir,
                set_time_index=(data_type == 'position')
            )
            
            # Trim if requested
            if trim_days is not None and len(df) > 0:
                if data_type == 'rfid':
                    start_time = df['chunk_start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['chunk_start'] < end_time]
                if data_type == 'foraging':
                    start_time = df['start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['start'] < end_time]
                if data_type == 'position':
                    start_time = df.index.min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df.loc[df.index < end_time]
                
                print(f"  Trimmed to {trim_days} days: {len(df)} records")
            
            # Store in result
            key = f"{period}_{data_type}"
            result[key] = df
            
            # For position data, handle duplicates
            if data_type == 'position' and len(df) > 0:
                original_len = len(df)
                df = df.reset_index()
                df = df.drop_duplicates(subset=['time', 'identity_name'])
                df = df.set_index('time')
                result[key] = df
                if len(df) < original_len:
                    print(f"  Removed duplicates: {original_len} -> {len(df)}")
    
    return result

# Load and prepare data

In [ ]:
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['social'],
    data_types=["patchinfo", "positiondenoised"]
)

social_patchinfo_df = data['social_patchinfo']
social_position_df = data['social_positiondenoised']
social_position_df.set_index('time', inplace=True)
social_position_df.sort_index(inplace=True)

In [ ]:
"""Find unique patch configurations and their counts"""

# Group by block_start and aggreGate patch_name and patch_rate combinations
config_df = social_patchinfo_df.groupby('block_start').apply(
    lambda x: tuple(sorted(zip(x['patch_name'], x['patch_rate']), key=lambda item: item[0])),
    include_groups=False
).reset_index(name='config')

# Count occurrences and sort by patch rates
config_counts = config_df['config'].value_counts().sort_index(
    key=lambda idx: idx.map(lambda x: (x[0][1], x[1][1], x[2][1]))
)
n_configs = len(config_counts)

# Create numbered mapping and add to dataframe
config_mapping = {config: i+1 for i, config in enumerate(config_counts.index)}
config_df['config_num'] = config_df['config'].map(config_mapping)

# Display results
print(f"Number of unique configs: {n_configs}\n")
print("Config#  Count  Patch1   Patch2   Patch3")
print("-" * 45)
for config, count in config_counts.items():
    rates = [f"{rate:.4f}" for _, rate in config]
    print(f"  {config_mapping[config]:2d}     {count:2d}    {rates[0]}   {rates[1]}   {rates[2]}")

display(config_df)

In [ ]:
"""Add block_start and config_num to social_position_df"""

# Merge config_num and block_start based on block_start times
merge_result = pd.merge_asof(
    social_position_df.reset_index()[['time']],
    config_df[['block_start', 'config_num']].sort_values('block_start'),
    left_on='time',
    right_on='block_start',
    direction='backward'
)

social_position_df['config_num']   = merge_result['config_num'].values
social_position_df['block_start']  = merge_result['block_start'].values

display(social_position_df)

In [ ]:
"""Make state df"""

# Choose focal mouse (0 or 1)
focal_identity = social_position_df['identity_name'].unique()[0]  # or [1]

# Split into self and other, keeping only what we need (include time + context)
self_df = social_position_df[social_position_df['identity_name'] == focal_identity]\
    .reset_index()[['time', 'experiment_name', 'block_start', 'config_num', 'x', 'y']]\
    .rename(columns={'x': 'x_self', 'y': 'y_self'})

other_df = social_position_df[social_position_df['identity_name'] != focal_identity]\
    .reset_index()[['time', 'experiment_name', 'block_start', 'x', 'y']]\
    .rename(columns={'x': 'x_other', 'y': 'y_other'})

# Exact merge on time + experiment + block (no asof, no tolerance)
state_df = self_df.merge(
    other_df,
    on=['time', 'experiment_name', 'block_start'],
    how='inner'
)

# Add velocities (finite differences within each experiment/block)
dt = 1.0 / fps
state_df[['vx_self', 'vy_self']] = state_df.groupby(
    ['experiment_name', 'block_start']
)[['x_self', 'y_self']].diff() / dt
state_df[['vx_other', 'vy_other']] = state_df.groupby(
    ['experiment_name', 'block_start']
)[['x_other', 'y_other']].diff() / dt

# Relative position and velocity (other - self)
state_df['dx']  = state_df['x_other'] - state_df['x_self']
state_df['dy']  = state_df['y_other'] - state_df['y_self']
state_df['dvx'] = state_df['vx_other'] - state_df['vx_self']
state_df['dvy'] = state_df['vy_other'] - state_df['vy_self']

# Drop the first frame of each block where velocities are NaN
state_df = state_df.dropna(subset=['vx_self', 'vy_self', 'vx_other', 'vy_other'])

display(state_df)

In [ ]:
"""Build transitions (s, a, s_next, done, c) from state_df"""

k = 10  # Δt = 200 ms at 50 Hz

state_cols = ["x_self", "y_self", "vx_self", "vy_self", "dx", "dy", "dvx", "dvy"]

# Index within each block and block length
state_df["block_idx"] = state_df.groupby(["experiment_name", "block_start"]).cumcount()
state_df["block_len"] = state_df.groupby(["experiment_name", "block_start"])["block_idx"].transform("max") + 1

# Next-state columns via k-step shift within each block
group_keys = ["experiment_name", "block_start"]
for col in state_cols:
    state_df[f"{col}_next"] = state_df.groupby(group_keys)[col].shift(-k)

# Keep only rows where a next state exists
mask = state_df["x_self_next"].notna()
trans_df = state_df[mask]

# s_t and s_{t+k}
s = trans_df[state_cols].to_numpy(dtype="float32")
s_next = trans_df[[f"{c}_next" for c in state_cols]].to_numpy(dtype="float32")

# k-step action: displacement of self
a = np.stack([
    trans_df["x_self_next"].values - trans_df["x_self"].values,
    trans_df["y_self_next"].values - trans_df["y_self"].values,
], axis=1).astype("float32")

# done: last valid transition in each block (after which no further k-step exists)
done = (trans_df["block_idx"] + k == trans_df["block_len"] - 1).to_numpy()

# Environment config (config_num at time t)
c = trans_df["config_num"].to_numpy(dtype="int64")
n_configs = c.max()
c_onehot = np.eye(n_configs)[c - 1].astype("float32")

print("Transitions:", len(s), "  States dim:", s.shape[1], "  Action dim:", a.shape[1])

In [ ]:
"""Prepare data for imitation policy training"""

# Training parameters - adjust as needed
batch_size = 2048
epochs = 2
additional_epochs = 0 # if >0 and checkpoint exists, number of additional epochs to train
log_every = 500 # number of batches between logging

# Define device and check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

# Build phi, a low-dimensional subset of state features most relevant for action selection
phi_cols = ["x_self", "y_self", "vx_self", "vy_self", "dx", "dy"]
phi_np = trans_df[phi_cols].to_numpy(dtype="float32")

# Convert arrays to torch
phi_tensor = torch.from_numpy(phi_np).float()
a_tensor = torch.from_numpy(a).float()

# z-score normalisation
phi_mean, phi_std = phi_tensor.mean(0), phi_tensor.std(0) + 1e-6
a_mean, a_std = a_tensor.mean(0), a_tensor.std(0) + 1e-6

phi_norm = (phi_tensor - phi_mean) / phi_std
actions_norm = (a_tensor - a_mean) / a_std

# Add env config one-hot encoding to phi
c_onehot_tensor = torch.from_numpy(c_onehot).float()
phi_full = torch.cat([phi_norm, c_onehot_tensor], dim=1)

# Pack into a dataset
imitation_dataset = torch.utils.data.TensorDataset(
    phi_full, # [N, 6 + n_configs]
    actions_norm,  # [N, 2]
)

# Batching and shuffling
imitation_loader = torch.utils.data.DataLoader(
    imitation_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

In [ ]:
"""Define imitation policy network and optimiser"""

class ImitationPolicy(nn.Module):
    def __init__(self, input_dim=4, # φ(s) = (x_self, y_self, dx, dy)
                 hidden_dim=128, 
                 action_dim=2 # action = (Δx, Δy)
                 ):
        super().__init__()
        # 2-layer MLP: φ(s) → 128-dim hidden features
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        ) 
        # Final linear layer mapping hidden features → mean of Gaussian over actions
        self.mu_head = nn.Linear(hidden_dim, action_dim) 
        # Learnable log-standard-deviation for each action dimension (σ_x, σ_y)
        # Using a global std (state-independent) for stability
        # We store log σ (unconstrained real values) and exponentiate it in forward() (to ensure positivity)
        self.log_std = nn.Parameter(torch.zeros(action_dim)) 

    def forward(self, phi):
        # Compute hidden features h = MLP(φ)
        h = self.net(phi)
        # Predict the Gaussian mean μ(φ)
        mu = self.mu_head(h)
        # Clamp log σ to keep the Gaussian variance in a reasonable range.
        # This prevents the model from inflating σ (collapse to a huge, flat Gaussian).
        # exp(log_std) ensures σ > 0, since log σ is unconstrained.
        # log_std = torch.clamp(self.log_std, -3.0, 0.0) # σ in [0.05, 1.0]
        # std = torch.exp(log_std)
        std = torch.exp(self.log_std) # clamp removed but can be re-added if needed
        return mu, std

    def log_prob(self, phi, a):
        # Compute μ and σ for the given φ(s) (forward pass)
        mu, std = self(phi)
        # Define π(a | φ(s)) as an independent Normal over Δx and Δy
        dist = torch.distributions.Normal(mu, std) 
        # Log-prob per dimension → sum = joint log-prob
        # p(a∣s)=p(Δx∣s)p(Δy∣s) so logp(a∣s)=logp(Δx∣s)+logp(Δy∣s)
        return dist.log_prob(a).sum(-1)

# Instantiate imitation policy and optimiser
policy = ImitationPolicy(input_dim=phi_full.shape[1], action_dim=actions_norm.shape[1]).to(device)
optimiser = optim.Adam(policy.parameters(), lr=1e-3)

In [ ]:
"""Fit imitation policy"""

# Check if saved file exists
load = False
save_path = save_dir / "Aeon 3 social 0.2/imitation_policy_w_c_2.pt"
if save_path.exists():
    load = True

if load:
    print("Loading imitation policy from disk...")
    checkpoint = torch.load(save_path, map_location=device)
    policy.load_state_dict(checkpoint["policy_net"])
    optimiser.load_state_dict(checkpoint["policy_opt"])
    
    phi_mean = checkpoint["phi_mean"]
    phi_std = checkpoint["phi_std"]
    a_mean = checkpoint["a_mean"]
    a_std = checkpoint["a_std"]

    start_epoch = checkpoint.get("epoch", 0)

    if additional_epochs <= 0 :
        print(f"Model loaded (epoch {start_epoch}). No additional training requested, skipping training.")
        do_train = False
    else:
        total_epochs = start_epoch + additional_epochs
        print(
            f"Model loaded (epoch {start_epoch}). "
            f"Continuing training for {additional_epochs} more epochs "
            f"(up to epoch {total_epochs})."
        )
        do_train = True
else:
    print("No existing checkpoint, training from scratch...")
    start_epoch = 0
    total_epochs = epochs
    do_train = True
    
if do_train:
    print("Training imitation policy...")

    # Train π(a | φ(s)) via maximum likelihood estimation (MLE)
    policy.train()
    for epoch in range(start_epoch, total_epochs):
        running_loss = 0.0
        n_batches = 0
        for step, (phi_batch, a_batch) in enumerate(tqdm(imitation_loader, desc=f"Epoch {epoch+1}", miniters=100)):
            # Move to GPU (or CPU)
            phi_batch = phi_batch.to(device, non_blocking=True) # [batch_size, 4]
            a_batch   = a_batch.to(device, non_blocking=True) # [batch_size, 2]

            # Compute log-probabilities and loss
            logp = policy.log_prob(phi_batch, a_batch) # [B]
            loss = -logp.mean() # maximise log-likelihood

            # Gradient step
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()

            # Track loss to print progress
            running_loss += loss.item()
            n_batches += 1

            if (step + 1) % log_every == 0:
                print(
                    f"Epoch {epoch+1}, Step {step+1}: "
                    f"loss={running_loss / n_batches:.4f}"
                )

        print(
            f"\n\nEpoch {epoch+1} done."
            f"loss={running_loss / n_batches:.4f}"
        )
        print("---")

        # Save checkpoint after each epoch
        checkpoint = {
            "policy_net": policy.state_dict(),
            "policy_opt": optimiser.state_dict(),
            "phi_cols": phi_cols,
            "phi_mean": phi_mean,
            "phi_std": phi_std,
            "a_mean": a_mean,
            "a_std": a_std,
            "epoch": epoch + 1,
        }

        # Always keep a "latest" checkpoint
        torch.save(checkpoint, save_path)

        # Also save a per-epoch checkpoint
        epoch_path = save_path.with_name(save_path.stem + f"_epoch{epoch+1}.pt")
        torch.save(checkpoint, epoch_path)
        print(f"Saved epoch {epoch+1} checkpoint to {epoch_path}")
    
    print(f"Saved final imitation policy to {save_path}")

In [ ]:
"""Compute rewards r(s,a) = log π(a | φ(s)) for all transitions"""

# Put network in inference mode (disables training-specific behaviour like dropout/batchnorm)
policy.eval()

# Process in batches to avoid OOM
batch_size = 8192
n_samples = phi_full.shape[0]
logp_list = []

# Disable gradient tracking (no graph → faster + lower memory)
with torch.no_grad():
    for i in tqdm(range(0, n_samples, batch_size), desc="Computing log probs"):
        # Get batch slice
        end_idx = min(i + batch_size, n_samples)
        phi_tensor = phi_full[i:end_idx].to(device) # [batch_size, 6 + n_configs]
        a_tensor = actions_norm[i:end_idx].to(device) # [batch_size, 2]
        # Compute log probs for this batch
        logp_batch = policy.log_prob(phi_tensor, a_tensor) # [batch_size] log π(a|φ(s))
        # Move to CPU immediately to free GPU memory
        logp_list.append(logp_batch.cpu().numpy())

# Concatenate all batches
r = np.concatenate(logp_list).astype("float32")

# Sanity check
print("\nReward stats for sanity check:")
print(f"max: {r.max():.3f} (expected ≈ -0.5 to -2)")
print(f"mean: {r.mean():.3f} (expected ≈ -2 to -4)")
print(f"std: {r.std():.3f} (expected ≈ 5–15)")
print(f"min: {r.min():.1f} (very negative outliers are normal)")

print("\nNumerical issues:")
print(f"NaN: {np.isnan(r).sum()} (should be 0)")
print(f"inf: {np.isinf(r).sum()} (should be 0)")

log_std=policy.log_std.detach().cpu().numpy()
std=np.exp(log_std)
print("\nPolicy std:")
print(f"log_std: {log_std} (expected in [-3,0])")
print(f"std: {std} (≈1 expected since actions were normalised)")

In [ ]:
"""Load metadata and define arena boundaries"""

exp, acquisition_computer = experiment["name"].split('-', 1)
acquisition_computer = acquisition_computer.upper()
root_path = f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{exp}"
metadata_reader = social02.Metadata
metadata = aeon_api.load(root_path, metadata_reader)['metadata'].iloc[0]
inner_radius = float(metadata.ActiveRegion.ArenaInnerRadius)
outer_radius = float(metadata.ActiveRegion.ArenaOuterRadius)
center_x = float(metadata.ActiveRegion.ArenaCenter.X)
center_y = float(metadata.ActiveRegion.ArenaCenter.Y)
nest_corner_1 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[0]
nest_corner_2 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[1]
nest_corner_3 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[2]
nest_corner_4 = metadata.ActiveRegion.NestRegion.ArrayOfPoint[3]
patch1_corner_1 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[0]
patch1_corner_2 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[1]
patch1_corner_3 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[2]
patch1_corner_4 = metadata.ActiveRegion.Patch1Region.ArrayOfPoint[3]
patch2_corner_1 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[0]
patch2_corner_2 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[1]
patch2_corner_3 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[2]
patch2_corner_4 = metadata.ActiveRegion.Patch2Region.ArrayOfPoint[3]
patch3_corner_1 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[0]
patch3_corner_2 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[1]
patch3_corner_3 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[2]
patch3_corner_4 = metadata.ActiveRegion.Patch3Region.ArrayOfPoint[3]

def get_validity_mask(xx, yy, metadata):
    # Extract geometry
    inner_radius = float(metadata.ActiveRegion.ArenaInnerRadius)
    outer_radius = float(metadata.ActiveRegion.ArenaOuterRadius)
    center_x = float(metadata.ActiveRegion.ArenaCenter.X)
    center_y = float(metadata.ActiveRegion.ArenaCenter.Y)
    
    # Nest geometry
    nest_pts = metadata.ActiveRegion.NestRegion.ArrayOfPoint
    nest_xs = [float(p.X) for p in nest_pts]
    nest_ys = [float(p.Y) for p in nest_pts]
    
    # Calculate distances
    dx = xx - center_x
    dy = yy - center_y
    r = np.sqrt(dx**2 + dy**2)
    
    # Define zones
    mask_inner = (r <= inner_radius)
    mask_corridor = (r >= inner_radius) & (r <= outer_radius)
    mask_nest = (
        (xx >= min(nest_xs)) & (xx <= max(nest_xs)) & 
        (yy >= min(nest_ys)) & (yy <= max(nest_ys))
    )
    
    return mask_inner | mask_corridor | mask_nest

def build_general_dashboard(
    model,
    config_counts,
    input_mean,       # phi_mean or s_mean
    input_std,        # phi_std or s_std
    locations,
    arena_img_array,
    output_mean=None, # a_mean (only needed for vector mode)
    output_std=None,  # a_std (only needed for vector mode)
    scalar_vmin=None, # Min value for heatmap color scale
    scalar_vmax=None, # Max value for heatmap color scale
    mode='vector',    # 'vector' for arrows, 'scalar' for heatmap
    feature_dim=6,    # 6 for imitation, 8 for RL/Nav
):
    
    # Setup background
    img_h, img_w = arena_img_array.shape[:2]
    pil_img = Image.fromarray((arena_img_array * 255).astype('uint8'))
    buff = io.BytesIO()
    pil_img.save(buff, format="PNG")
    img_str = base64.b64encode(buff.getvalue()).decode("utf-8")
    img_src = "data:image/png;base64," + img_str

    # Initial Traces
    data_traces = []
    
    # Trace 0: The Data (Vector or Scalar)
    if mode == 'vector':
        trace_main = go.Scatter(
            x=[], y=[],
            mode='lines',
            line=dict(color='blue', width=2), 
            name='Policy',
            hoverinfo='skip'
        )
    else:
        # Heatmap for Value/Scalar fields
        trace_main = go.Heatmap(
            z=[], x=[], y=[],
            colorscale='Viridis',
            opacity=0.6,
            showscale=True,
            zmin=scalar_vmin,
            zmax=scalar_vmax,
            name='Value',
            hoverinfo='z'
        )
    data_traces.append(trace_main)
    
    # Trace 1: The Other Mouse
    trace_other = go.Scatter(
        x=[], y=[],
        mode='markers',
        marker=dict(size=15, color='red', symbol='star'),
        name='Other'
    )
    data_traces.append(trace_other)

    fig = go.FigureWidget(data=data_traces)
    
    fig.update_layout(
        title=f"{mode.capitalize()} Explorer",
        width=900, height=800,
        xaxis=dict(range=[0, img_w], showgrid=False, zeroline=False),
        yaxis=dict(range=[0, img_h], showgrid=False, zeroline=False, scaleanchor='x'),
        template='plotly_white',
        margin=dict(l=20, r=20, t=30, b=20),
        images=[dict(
            source=img_src,
            xref="x", yref="y",
            x=0, y=img_h,
            sizex=img_w, sizey=img_h,
            sizing="stretch",
            opacity=0.5,
            layer="below"
        )]
    )

    # UI Controls
    config_opts = []
    for i, cfg_tuple in enumerate(config_counts.index):
        rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
        label = f"({', '.join(rates)})"
        config_opts.append((label, i))

    w_config = widgets.Dropdown(options=config_opts, value=0, description='Config:', layout=widgets.Layout(width='300px'))
    w_location = widgets.Dropdown(options=locations.keys(), value=list(locations.keys())[0], description='Other:', layout=widgets.Layout(width='250px'))
    w_density = widgets.IntSlider(value=25, min=10, max=60, step=5, description='Density:', layout=widgets.Layout(width='300px'))
    
    # Scale slider (only relevant for vector plots)
    w_scale = widgets.FloatSlider(value=5.0, min=1.0, max=20.0, step=1.0, description='Arrow Scale:', layout=widgets.Layout(width='300px'))
    if mode == 'scalar':
        w_scale.layout.display = 'none'

    w_vx = widgets.FloatSlider(value=0, min=-50, max=50, step=5, description='Self Vx', readout_format='.0f', layout=widgets.Layout(width='300px'))
    w_vy = widgets.FloatSlider(value=0, min=-50, max=50, step=5, description='Self Vy', readout_format='.0f', layout=widgets.Layout(width='300px'))
    w_btn = widgets.Button(description='Update', button_style='info', icon='refresh')

    # Logic
    def get_arrow_geometry(x_start, y_start, u, v, scale, arrow_len=10, arrow_angle=0.4):
        x_end = x_start + u * scale
        y_end = y_start + v * scale
        theta = np.arctan2(v, u) 
        
        x_wing1 = x_end - arrow_len * np.cos(theta + arrow_angle)
        y_wing1 = y_end - arrow_len * np.sin(theta + arrow_angle)
        x_wing2 = x_end - arrow_len * np.cos(theta - arrow_angle)
        y_wing2 = y_end - arrow_len * np.sin(theta - arrow_angle)
        
        N = len(x_start)
        x_plot = np.empty(N * 9)
        y_plot = np.empty(N * 9)
        x_plot[:] = np.nan
        y_plot[:] = np.nan
        
        x_plot[0::9] = x_start; x_plot[1::9] = x_end; y_plot[0::9] = y_start; y_plot[1::9] = y_end
        x_plot[3::9] = x_end; x_plot[4::9] = x_wing1; y_plot[3::9] = y_end; y_plot[4::9] = y_wing1
        x_plot[6::9] = x_end; x_plot[7::9] = x_wing2; y_plot[6::9] = y_end; y_plot[7::9] = y_wing2
        return x_plot, y_plot

    def update_chart(_):
        # Define device and check for GPU
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", device)
        if device.type == "cuda":
            print("GPU name:", torch.cuda.get_device_name(0))

        # Regenerate Grid
        # For heatmaps, we might want higher density than arrows
        density_mult = 3 if mode == 'scalar' else 1 
        nx = ny = w_density.value * density_mult
        
        xs = np.linspace(0, img_w, nx)
        ys = np.linspace(0, img_h, ny)
        xx, yy = np.meshgrid(xs, ys)
        
        cfg_idx = w_config.value
        other_x, other_y = locations[w_location.value]
        vx, vy = w_vx.value, w_vy.value
        scale_val = w_scale.value

        valid_mask = get_validity_mask(xx, yy, metadata)
        grid_points = np.stack([xx.ravel(), yy.ravel()], axis=1)
        
        # Build Input Features
        # Both scenarios use 0=x, 1=y, 4=dx, 5=dy. 
        # We assume 2=vx, 3=vy for mapping the sliders.
        feat_grid = np.zeros((len(grid_points), feature_dim), dtype="float32")
        feat_grid[:, 0] = grid_points[:, 0] # x
        feat_grid[:, 1] = grid_points[:, 1] # y
        feat_grid[:, 2] = vx                # vx
        feat_grid[:, 3] = vy                # vy
        feat_grid[:, 4] = other_x - grid_points[:, 0] # dx
        feat_grid[:, 5] = other_y - grid_points[:, 1] # dy
        
        feat_tensor = torch.from_numpy(feat_grid).to(device)
        feat_norm = (feat_tensor - input_mean.to(device)) / input_std.to(device)
        
        c_grid = np.zeros((len(grid_points), len(config_counts)), dtype="float32")
        c_grid[:, cfg_idx] = 1.0
        feat_full = torch.cat([feat_norm, torch.from_numpy(c_grid).to(device)], dim=1)
        
        # Inference & Update
        if mode == 'vector':
            model.eval()
            with torch.no_grad():
                mu, _ = model(feat_full)
                actions = (mu.cpu().numpy() * output_std.detach().cpu().numpy()) + output_mean.detach().cpu().numpy()
            
            u, v = actions[:, 0], actions[:, 1]
            
            # Filter valid points
            valid_idx = np.where(valid_mask.ravel())[0]
            x_v = grid_points[valid_idx, 0]
            y_v = grid_points[valid_idx, 1]
            u_v = u[valid_idx]
            v_v = v[valid_idx]
            
            x_lines, y_lines = get_arrow_geometry(x_v, y_v, u_v, v_v, scale=scale_val)
            
            with fig.batch_update():
                fig.data[0].x = x_lines
                fig.data[0].y = y_lines
                fig.data[1].x = [other_x]
                fig.data[1].y = [other_y]
                
        else: # scalar/heatmap
            model.eval()
            with torch.no_grad():
                vals = model(feat_full).cpu().numpy()
            
            # Reshape to grid
            z_grid = vals.reshape(ny, nx)
            
            # Apply mask (set invalid to None for transparency in Plotly)
            z_masked = np.where(valid_mask, z_grid, None)
            
            with fig.batch_update():
                fig.data[0].z = z_masked
                fig.data[0].x = xs
                fig.data[0].y = ys
                fig.data[1].x = [other_x]
                fig.data[1].y = [other_y]

    w_btn.on_click(update_chart)
    update_chart(None)
    
    # Layout
    row1 = widgets.HBox([w_config, w_location])
    row2 = widgets.HBox([w_density, w_scale])
    row3 = widgets.HBox([w_vx, w_vy, w_btn])
    
    return widgets.VBox([
        widgets.HTML(f"<b>{mode.capitalize()} Controls</b>"),
        row1, row2, row3, 
        fig
    ])

In [ ]:
"""Plot Imitation Policy for different fixed other positions"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}
n_locs = len(locations)

# Load image and determine global bounds
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h]

# Setup grid
nx, ny = 30, 30
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Prepare figure
fig, axes = plt.subplots(
    n_configs, n_locs,
    figsize=(4 * n_locs, 4 * n_configs),
    squeeze=False,
    dpi=300 
)

# Loop over configs (rows) and locations (columns)
unique_configs = list(config_counts.index)
for cfg_idx, cfg_tuple in enumerate(unique_configs):
    rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
    # Format: (Rate1, Rate2, Rate3)
    cfg_name = f"({rates[0]}, {rates[1]}, {rates[2]})"

    # One-hot encoding for config
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    for loc_j, (loc_name, (other_x, other_y)) in enumerate(locations.items()):
        ax = axes[cfg_idx, loc_j]
        
        print(f"Computing imitation policy for config {cfg_name} with fixed other at {loc_name}...")

        # Construct Input (phi = [x, y, dx, dy])
        phi_grid = np.zeros((nx * ny, 6), dtype="float32")
        phi_grid[:, 0] = xx.ravel() # x_self
        phi_grid[:, 1] = yy.ravel() # y_self
        phi_grid[:, 2] = 0
        phi_grid[:, 3] = 0
        phi_grid[:, 4] = other_x - xx.ravel() # dx
        phi_grid[:, 5] = other_y - yy.ravel() # dy
        
        # Normalize (using phi stats)
        phi_tensor = torch.from_numpy(phi_grid).to(device)
        phi_norm = (phi_tensor - phi_mean.to(device)) / phi_std.to(device)

        # Add env config one-hot encoding
        phi_full = torch.cat([phi_norm, c_grid_tensor], dim=1)

        # Query Imitation Policy
        policy.eval()
        with torch.no_grad():
            mu, _ = policy(phi_full)
            mu = mu.cpu().numpy()

        # Un-normalize actions
        a_std_np = a_std.detach().cpu().numpy()
        a_mean_np = a_mean.detach().cpu().numpy()
        actions = (mu * a_std_np) + a_mean_np

        u = actions[:, 0].reshape(ny, nx)
        v = actions[:, 1].reshape(ny, nx)

        # Apply masking
        valid_start = get_validity_mask(xx, yy, metadata)
        u = np.where(valid_start, u, np.nan)
        v = np.where(valid_start, v, np.nan)

        # Plot background
        if arena_img is not None:
            ax.imshow(arena_img, origin="lower", alpha=0.5, extent=extent_img)

        # Plot self movement
        ax.quiver(xx, yy, u, v, color='blue', scale=None, scale_units='inches')

        # Plot fixed other mouse
        ax.plot(other_x, other_y, 'b*', markersize=25, markeredgecolor='white', label=f"Other ({loc_name})")

        # Labeling Logic:
        # Column Labels (Top Row only)
        if cfg_idx == 0:
            ax.set_title(loc_name, fontsize=14, fontweight='bold')
        else:
            ax.set_title("") # Clear title for other rows

        # Row Labels (Left Column only)
        if loc_j == 0:
            # Rotated 90 degrees, adjusted labelpad to 20 since it's vertical now
            ax.set_ylabel(f"Config\n{cfg_name}", fontsize=12, fontweight='bold', rotation=90, labelpad=20)
            # Remove standard y ticks to clean up
            ax.set_yticks([]) 
        else:
            ax.set_ylabel("")
            ax.set_yticks([])

        # Remove x ticks for all plots to clean up
        ax.set_xticks([])
        
        # Lock view to full arena dimensions
        ax.set_xlim(0, img_w)
        ax.set_ylim(0, img_h)
        
        ax.set_aspect('equal')

# Add a single legend for the whole figure
# We create a dummy handle to represent the "Other" mouse star
legend_elements = [Line2D([0], [0], marker='*', color='w', label='Other Mouse',
                          markerfacecolor='b', markersize=15)]
fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
"""Interactive Gaussian Policy Distribution Viewer"""

# P(a|s) = N(Δx,Δy; μ(s), σ) overlaid on a cropped arena image.

# Tunable parameters
_SMALL_R = 15
_MOUSE_PAD = 20
_GAUSS_PAD = 20
_MIN_SAFE_HALF_R = 60

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}

# Load image
_arena_img = plt.imread("arena.png")
_img_h, _img_w = _arena_img.shape[:2]

# Define shared locations
_nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
_nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
_p1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
_p1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
_p2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
_p2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
_p3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
_p3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
_corr_r = (inner_radius + outer_radius) / 2

_shared_locations = {
    "Arena Center": (center_x, center_y),
    "Nest": (_nest_x, _nest_y),
    "Gate": (center_x - _corr_r, center_y),
    "Patch 1": (_p1_x, _p1_y),
    "Patch 2": (_p2_x, _p2_y),
    "Patch 3": (_p3_x, _p3_y),
}

# Add extra self-only positions
_nest_ang = np.arctan2(_nest_y - center_y, _nest_x - center_x)
_corr_nest = (center_x + _corr_r * np.cos(_nest_ang),
               center_y + _corr_r * np.sin(_nest_ang))
_inner_gate = (center_x - 0.7 * inner_radius, center_y)

_self_locations = dict(_shared_locations)
_self_locations["Corridor–Nest side"] = _corr_nest
_self_locations["Inner arena–Gate"] = _inner_gate
for _pname, _px, _py in [("Patch 1", _p1_x, _p1_y),
                         ("Patch 2", _p2_x, _p2_y),
                         ("Patch 3", _p3_x, _p3_y)]:
    for _d, _ox, _oy in [("N", 0, 1), ("S", 0, -1), ("E", 1, 0), ("W", -1, 0)]:
        _self_locations[f"Near {_pname} ({_d})"] = (_px + _SMALL_R * _ox, _py + _SMALL_R * _oy)

# Define config options
_unique_configs = list(config_counts.index)
_cfg_options = [(str(cfg), i) for i, cfg in enumerate(_unique_configs)]

# Store normalization stats
_phi_mean_np = phi_mean.detach().cpu().numpy()
_phi_std_np = phi_std.detach().cpu().numpy()
_a_mean_np = a_mean.detach().cpu().numpy()
_a_std_np = a_std.detach().cpu().numpy()

# Create widgets
_lstyle = {'description_width': '80px'}
w_self = widgets.Dropdown(options=list(_self_locations.keys()), value="Arena Center",
                          description="Self:", style=_lstyle)
w_other = widgets.Dropdown(options=list(_shared_locations.keys()), value="Patch 1",
                            description="Other:", style=_lstyle)
w_cfg = widgets.Dropdown(options=_cfg_options, value=0, description="Config:", style=_lstyle)
w_btn = widgets.Button(description="Update", button_style="info", icon="refresh")
w_save = widgets.Button(description="Save SVG", button_style="success", icon="download")

_out = widgets.Output()
_current_fig = {}

def _update(_):
    self_x, self_y = _self_locations[w_self.value]
    other_x, other_y = _shared_locations[w_other.value]
    s_raw = np.array([self_x, self_y, 0.0, 0.0,
                      other_x - self_x, other_y - self_y], dtype="float32")
    s_norm = (s_raw - _phi_mean_np) / _phi_std_np

    cfg_idx = w_cfg.value
    c_vec = np.zeros(n_configs, dtype="float32")
    c_vec[cfg_idx] = 1.0
    phi_t = torch.tensor(np.concatenate([s_norm, c_vec]),
                         dtype=torch.float32).unsqueeze(0).to(device)

    policy.eval()
    with torch.no_grad():
        mu_norm_t, std_t = policy(phi_t)
    mu = mu_norm_t.squeeze().cpu().numpy() * _a_std_np + _a_mean_np
    sigma = std_t.cpu().numpy() * _a_std_np

    mu_arena = np.array([self_x + mu[0], self_y + mu[1]])

    # Square crop sized by the larger of mouse separation or Gaussian footprint
    mouse_sep_r = 0.5 * np.hypot(other_x - self_x, other_y - self_y) + _MOUSE_PAD
    gaussian_r = 4 * sigma.max() + _GAUSS_PAD
    half_r = max(mouse_sep_r, gaussian_r, _MIN_SAFE_HALF_R)
    _x_pts = np.array([self_x, other_x, mu_arena[0]])
    _y_pts = np.array([self_y, other_y, mu_arena[1]])
    crop_cx = 0.5 * (_x_pts.min() + _x_pts.max())
    crop_cy = 0.5 * (_y_pts.min() + _y_pts.max())
    x_lo = max(crop_cx - half_r, 0)
    x_hi = min(crop_cx + half_r, _img_w)
    y_lo = max(crop_cy - half_r, 0)
    y_hi = min(crop_cy + half_r, _img_h)

    # Gaussian grid in arena coords
    X, Y = np.meshgrid(np.linspace(x_lo, x_hi, 300),
                       np.linspace(y_lo, y_hi, 300))
    Z = multivariate_normal(mean=mu_arena, cov=np.diag(sigma**2)).pdf(np.dstack([X, Y]))

    # Hide very low-density tails so the arena still shows through
    _z_floor = Z.max() * 0.01
    _Z_masked = np.ma.masked_less(Z, _z_floor)

    with _out:
        _out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7.2, 4.8))
        fig.subplots_adjust(right=0.72)
        _current_fig['fig'] = fig

        # Plot background
        ax.imshow(_arena_img, origin='lower', extent=[0, _img_w, 0, _img_h], alpha=0.8)

        # Plot heatmap
        _heat = ax.imshow(
            _Z_masked,
            origin='lower',
            extent=[x_lo, x_hi, y_lo, y_hi],
            cmap='viridis',
            interpolation='bilinear',
            alpha=0.82,
            vmin=_z_floor,
            vmax=Z.max(),
            zorder=2,
        )
        _cbar = fig.colorbar(_heat, ax=ax, fraction=0.046, pad=0.03)
        _cbar.set_label('Probability density', fontsize=10)
        _cbar.ax.tick_params(labelsize=8)

        # Plot mean crosshair
        _ch = 2 * sigma.max()
        ax.plot([mu_arena[0]-_ch, mu_arena[0]+_ch], [mu_arena[1], mu_arena[1]],
                color='gold', ls='--', lw=1.5, zorder=4, label='μ (mean action)')
        ax.plot([mu_arena[0], mu_arena[0]], [mu_arena[1]-_ch, mu_arena[1]+_ch],
                color='gold', ls='--', lw=1.5, zorder=4)

        ax.plot(self_x,  self_y,  'k*', markersize=9, zorder=5, label='self')
        ax.plot(other_x, other_y, 'r*', markersize=9, zorder=5, label='other')

        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(y_lo, y_hi)
        ax.set_aspect('equal')
        ax.set_xlabel('x  (px)', fontsize=11)
        ax.set_ylabel('y  (px)', fontsize=11)
        ax.set_title(f"Self: {w_self.value}  |  Other: {w_other.value}", fontsize=9)
        ax.legend(bbox_to_anchor=(1.28, 1), loc='upper left',
                  borderaxespad=0, fontsize=9, framealpha=0.6)
        ax.spines[['top', 'right']].set_visible(False)
        plt.show()

def _save_svg(_):
    fig = _current_fig.get('fig')
    if fig is not None:
        fig.savefig('gaussian_policy_fig.svg', bbox_inches='tight')
        print('Saved: gaussian_policy_fig.svg')
    else:
        print('No figure yet — click Update first.')

w_btn.on_click(_update)
w_save.on_click(_save_svg)

display(
    widgets.VBox([
        widgets.HBox([w_self, w_other]),
        widgets.HBox([w_cfg,  w_btn,  w_save]),
    ]),
    _out,
)
_update(None)


In [ ]:
# dashboard = build_general_dashboard(
#     model=policy,
#     config_counts=config_counts,
#     input_mean=phi_mean,
#     input_std=phi_std,
#     output_mean=a_mean,
#     output_std=a_std,
#     locations=locations,
#     arena_img_array=arena_img,
#     mode='vector',
#     feature_dim=6
# )
# display(dashboard)

In [ ]:
"""Prepare data for IQL"""

# Training parameters - adjust as needed
batch_size = 4096
epochs = 6
additional_epochs = 0 # if >0 and checkpoint exists, number of additional epochs to train
log_every = 500 # number of batches between logging

# Define device and check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

# Convert arrays to torch
s_tensor = torch.from_numpy(s).float()
a_tensor = torch.from_numpy(a).float()
r_tensor = torch.from_numpy(r).float()
s_next_tensor = torch.from_numpy(s_next).float()
done_tensor = torch.from_numpy(done.astype("float32"))

# z-score normalisation
s_mean, s_std = s_tensor.mean(0), s_tensor.std(0) + 1e-6
a_mean, a_std = a_tensor.mean(0), a_tensor.std(0) + 1e-6

s_norm = (s_tensor - s_mean) / s_std
s_next_norm = (s_next_tensor - s_mean) / s_std
a_norm = (a_tensor - a_mean) / a_std

# Add env config one-hot encoding to s
c_onehot_tensor = torch.from_numpy(c_onehot).float()
s_full = torch.cat([s_norm, c_onehot_tensor], dim=1)
s_next_full = torch.cat([s_next_norm, c_onehot_tensor], dim=1)

# Pack into a dataset
iql_dataset = torch.utils.data.TensorDataset(
    s_full, # states [N, 8 + n_configs]
    a_norm, # actions [N, 2]
    r_tensor.unsqueeze(-1), # rewards [N, 1]
    s_next_full, # next states [N, 8 + n_configs]
    done_tensor.unsqueeze(-1) # done flags [N, 1]
)

# Batching and shuffling
iql_loader = torch.utils.data.DataLoader(
    iql_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

In [ ]:
"""Define IQL networks and optimisers"""

# Network parameters - adjust as needed
state_dim = s_full.shape[1]
action_dim = a_norm.shape[1]
hidden_dim = 256
gamma = 0.99 # discount factor
tau_expectile = 0.7 # expectile parameter for value regression
beta = 3.0 # temperature for advantage weights
target_update_rate = 0.005 # Polyak update rate for target nets

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # 3-layer MLP
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

# Value network V(s)
class ValueNet(nn.Module):
    def __init__(self, state_dim, hidden_dim):
        super().__init__()
        self.body = MLP(state_dim, hidden_dim, 1)

    def forward(self, s):
        return self.body(s).squeeze(-1) # [batch_size]

# Q-value network Q(s,a)
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        self.body = MLP(state_dim + action_dim, hidden_dim, 1)

    def forward(self, s, a):
        sa = torch.cat([s, a], dim=-1)
        return self.body(sa).squeeze(-1) # [batch_size]

# Gaussian policy network π(a | s)
class GaussianPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim)) # global log σ

    def forward(self, s):
        h = self.net(s)
        mu = self.mu_head(h)
        std = torch.exp(self.log_std)
        return mu, std

    def log_prob(self, s, a):
        mu, std = self(s)
        dist = torch.distributions.Normal(mu, std)
        return dist.log_prob(a).sum(-1)  # [batch_size]

# Polyak averaging utility
# Called once per batch to keep the target net slowly tracking the online net
# Implements: θ_target ← (1 - τ) * θ_target + τ * θ_source
def polyak_update(source, target, tau):
    with torch.no_grad():
        # Update target network one parameter at a time
        for p, p_targ in zip(source.parameters(), target.parameters()): 
            p_targ.data.mul_(1 - tau).add_(tau * p.data)

# Instantiate networks
value_net = ValueNet(state_dim, hidden_dim).to(device)
# A slow-moving copy of value_net (the “target value network”)
# This gives a stable bootstrap term r + γ * V_target(s′) during Q-learning and prevents divergence.
value_target = ValueNet(state_dim, hidden_dim).to(device)
value_target.load_state_dict(value_net.state_dict())

# Two separate Q networks (the “double-Q” trick)
# Double-Q reduces overestimation bias by taking the conservative target min(Q1(s,a), Q2(s,a))
# This is standard in modern actor–critic RL
q1_net = QNet(state_dim, action_dim, hidden_dim).to(device)
q2_net = QNet(state_dim, action_dim, hidden_dim).to(device)

policy_net = GaussianPolicy(state_dim, action_dim, hidden_dim).to(device)

# Optimisers
value_opt = optim.Adam(value_net.parameters(), lr=3e-4) # only updates value_net, value_target is updated via Polyak averaging
q_opt = optim.Adam(list(q1_net.parameters()) + list(q2_net.parameters()), lr=3e-4) # for both q1 and q2_net
policy_opt = optim.Adam(policy_net.parameters(), lr=3e-4) # for policy_net

In [ ]:
"""Train IQL on offline dataset"""

# Check if saved file exists
load = False
save_path = save_dir / "Aeon 3 social 0.2/iql_model_w_c_1_epoch4.pt"
if save_path.exists():
    load = True

if load:
    print("Loading IQL model from disk...")
    checkpoint = torch.load(save_path, map_location=device)
    value_net.load_state_dict(checkpoint["value_net"])
    value_target.load_state_dict(checkpoint["value_target"])
    q1_net.load_state_dict(checkpoint["q1_net"])
    q2_net.load_state_dict(checkpoint["q2_net"])
    policy_net.load_state_dict(checkpoint["policy_net"])

    value_opt.load_state_dict(checkpoint["value_opt"])
    q_opt.load_state_dict(checkpoint["q_opt"])
    policy_opt.load_state_dict(checkpoint["policy_opt"])

    s_mean = checkpoint["s_mean"]
    s_std = checkpoint["s_std"]
    a_mean = checkpoint["a_mean"]
    a_std = checkpoint["a_std"]

    start_epoch = checkpoint.get("epoch", 0)

    if additional_epochs <= 0 :
        print(f"Model loaded (epoch {start_epoch}). No additional training requested, skipping training.")
        do_train = False
    else:
        total_epochs = start_epoch + additional_epochs
        print(
            f"Model loaded (epoch {start_epoch}). "
            f"Continuing training for {additional_epochs} more epochs "
            f"(up to epoch {total_epochs})."
        )
        do_train = True
else:
    print("No existing checkpoint, training from scratch...")
    start_epoch = 0
    total_epochs = epochs
    do_train = True
    
if do_train:
    print("Training IQL model...")

    value_net.train()
    q1_net.train()
    q2_net.train()
    policy_net.train()

    for epoch in range(start_epoch, total_epochs):
        running_v_loss = 0.0
        running_q_loss = 0.0
        running_pi_loss = 0.0
        n_batches = 0

        for step, (s_b, a_b, r_b, s_next_b, done_b) in enumerate(tqdm(iql_loader, desc=f"Epoch {epoch+1}", miniters=100)):
            # Move batch to device and fix shapes
            s_b = s_b.to(device) # [batch_size, state_dim]
            a_b = a_b.to(device) # [batch_size, action_dim]
            r_b = r_b.to(device).squeeze(-1) # [batch_size]
            s_next_b = s_next_b.to(device) # [batch_size, state_dim]
            done_b = done_b.to(device).squeeze(-1) # [batch_size]

            # Value update (expectile regression)
            # No gradients because in the V update, the Q-values act as fixed targets (teacher)
            with torch.no_grad():
                # Compute Q-values under both critics
                q1_val = q1_net(s_b, a_b)
                q2_val = q2_net(s_b, a_b)
                # Take min to reduce overestimation bias
                q_min = torch.min(q1_val, q2_val)  # [batch_size]

            # Update V(s) using expectile regression: we compare Q(s,a) vs V(s)
            # If Q > V then the action looks better-than-typical for that state → pull V upward strongly (weight = τ)
            # If Q < V then the action looks worse-than-typical → pull V downward weakly (weight = 1−τ)
            # This makes V track the upper tail of Q and ignore low-return actions
            v = value_net(s_b)  # [batch_size]
            diff = q_min - v
            weight = torch.where(diff >= 0, tau_expectile, 1.0 - tau_expectile) # Choose weight depending on sign of diff
            value_loss = (weight * diff.pow(2)).mean() # Weighted squared loss → expectile regression

            value_opt.zero_grad()
            value_loss.backward()
            value_opt.step()

            # Q update (Bellman regression with V_target)
            # Now Q is the student: we freeze the target value network inside no_grad,
            # and update q1/q2 to match this fixed TD target
            with torch.no_grad():
                v_next = value_target(s_next_b)
                # done_b = 1 at episode end → no bootstrap term there (target = r only)
                target = r_b + gamma * (1.0 - done_b) * v_next

            q1 = q1_net(s_b, a_b)
            q2 = q2_net(s_b, a_b)
            # Sum TD errors from both critics per sample, then average over batch
            # Could also divide by 2 to get the mean per critic; it just rescales the loss
            q_loss = ((q1 - target).pow(2) + (q2 - target).pow(2)).mean()

            q_opt.zero_grad()
            q_loss.backward()
            q_opt.step()

            # Policy update (advantage-weighted behaviour cloning)
            with torch.no_grad():
                q1_pi = q1_net(s_b, a_b)
                q2_pi = q2_net(s_b, a_b)
                q_min_pi = torch.min(q1_pi, q2_pi)
                v = value_net(s_b)
                adv = q_min_pi - v # [batch_size], advantage
                weights = torch.exp(adv / beta) # beta is temperature parameter: larger beta → softer weighting
                weights = torch.clamp(weights, max=20.0) # Clamp so high-advantage samples don’t blow up gradients

            log_pi = policy_net.log_prob(s_b, a_b) # [batch_size], probability of the dataset action under the current policy π(a|s)
            policy_loss = -(weights * log_pi).mean() # If adv is positive → weight positive → policy pushed to imitate that action and vice-versa

            policy_opt.zero_grad()
            policy_loss.backward()
            policy_opt.step()

            # Target network update
            polyak_update(value_net, value_target, target_update_rate)

            # Track losses
            running_v_loss += value_loss.item()
            running_q_loss += q_loss.item()
            running_pi_loss += policy_loss.item()
            n_batches += 1

            if (step + 1) % log_every == 0:
                print(
                    f"Epoch {epoch+1} Step {step+1}: "
                    f"V_loss={running_v_loss / n_batches:.4f}, "
                    f"Q_loss={running_q_loss / n_batches:.4f}, "
                    f"Pi_loss={running_pi_loss / n_batches:.4f}"
                )

        print(
            f"\n\nEpoch {epoch+1} done."
            f"V_loss={running_v_loss / n_batches:.4f}, "
            f"Q_loss={running_q_loss / n_batches:.4f}, "
            f"Pi_loss={running_pi_loss / n_batches:.4f}"
        )
        print("---")

        # Save checkpoint after each epoch
        checkpoint = {
            "value_net": value_net.state_dict(),
            "value_target": value_target.state_dict(),
            "q1_net": q1_net.state_dict(),
            "q2_net": q2_net.state_dict(),
            "policy_net": policy_net.state_dict(),
            "value_opt": value_opt.state_dict(),
            "q_opt": q_opt.state_dict(),
            "policy_opt": policy_opt.state_dict(),
            "s_mean": s_mean,
            "s_std": s_std,
            "a_mean": a_mean,
            "a_std": a_std,
            "gamma": gamma,
            "tau_expectile": tau_expectile,
            "beta": beta,
            "epoch": epoch + 1,
        }

        # Always keep a "latest" checkpoint
        torch.save(checkpoint, save_path)

        # Also save a per-epoch checkpoint
        epoch_path = save_path.with_name(save_path.stem + f"_epoch{epoch+1}.pt")
        torch.save(checkpoint, epoch_path)
        print(f"Saved epoch {epoch+1} checkpoint to {epoch_path}")
    
    print(f"Saved final IQL model to {save_path}")

In [ ]:
"""Plot value map for different fixed other positions"""

# Setup geometry and metadata
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}
n_locs = len(locations)

# Load image
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h]

# Setup grid
nx, ny = 200, 200
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Pre-calculate masks
valid_mask = get_validity_mask(xx, yy, metadata)

# Container for computed data and scale calculation
plot_data = []
all_valid_pixels = []

unique_configs = list(config_counts.index)

# First Pass: Compute all grids
for cfg_idx, cfg_tuple in enumerate(unique_configs):
    rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
    cfg_name = f"({rates[0]}, {rates[1]}, {rates[2]})"

    # One-hot encoding for config
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    for loc_j, (loc_name, (other_x, other_y)) in enumerate(locations.items()):
        print(f"Computing masked values for config {cfg_name} with fixed other at {loc_name}...")

        # Construct state
        s_grid = np.zeros((nx * ny, 8), dtype="float32")
        s_grid[:, 0] = xx.ravel() # x_self
        s_grid[:, 1] = yy.ravel() # y_self
        s_grid[:, 4] = other_x - xx.ravel() # dx
        s_grid[:, 5] = other_y - yy.ravel() # dy
        
        # Normalize and query
        s_tensor = torch.from_numpy(s_grid).to(device)
        s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)

        s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

        value_net.eval()
        with torch.no_grad():
            vals = value_net(s_full).cpu().numpy()
        
        # Reshape
        v_vals = vals.reshape(ny, nx)

        # Apply mask
        v_vals_masked = np.where(valid_mask, v_vals, np.nan)
        
        # Collect valid pixels for robust scaling
        valid_pixels = v_vals_masked[~np.isnan(v_vals_masked)]
        if len(valid_pixels) > 0:
            all_valid_pixels.append(valid_pixels)

        # Store for plotting
        plot_data.append({
            'cfg_idx': cfg_idx,
            'loc_j': loc_j,
            'cfg_name': cfg_name,
            'loc_name': loc_name,
            'other_x': other_x,
            'other_y': other_y,
            'grid': v_vals_masked
        })

# Compute robust scale (2nd and 98th percentiles)
if all_valid_pixels:
    concat_pixels = np.concatenate(all_valid_pixels)
    robust_vmin, robust_vmax = np.nanpercentile(concat_pixels, [2, 98])
else:
    robust_vmin, robust_vmax = 0, 1

# Prepare figure
fig, axes = plt.subplots(
    n_configs, n_locs, 
    figsize=(4 * n_locs, 4 * n_configs), 
    squeeze=False,
    dpi=300
)

# Second Pass: Plot using shared robust scale
for data in plot_data:
    cfg_idx = data['cfg_idx']
    loc_j = data['loc_j']
    ax = axes[cfg_idx, loc_j]
    
    # Plot background
    if arena_img is not None:
        ax.imshow(arena_img, origin="lower", alpha=1.0, extent=extent_img)

    # Plot masked values
    im = ax.imshow(
        data['grid'],
        origin="lower",
        extent=[0, img_w, 0, img_h],
        cmap="viridis",
        alpha=0.6,
        vmin=robust_vmin,
        vmax=robust_vmax
    )
    
    # Add colorbar
    fig.colorbar(im, ax=ax, label="Value V(s)", fraction=0.046, pad=0.04)

    # Plot fixed other mouse
    ax.plot(data['other_x'], data['other_y'], 'b*', markersize=25, markerfacecolor='none', markeredgecolor='white', label=f"Other ({data['loc_name']})")

    # Labeling Logic:
    # Column Labels (Top Row only)
    if cfg_idx == 0:
        ax.set_title(data['loc_name'], fontsize=14, fontweight='bold')
    else:
        ax.set_title("") 

    # Row Labels (Left Column only)
    if loc_j == 0:
        ax.set_ylabel(f"Config\n{data['cfg_name']}", fontsize=12, fontweight='bold', rotation=90, labelpad=20)
        ax.set_yticks([]) 
    else:
        ax.set_ylabel("")
        ax.set_yticks([])

    # Remove x ticks for all plots
    ax.set_xticks([])
    
    ax.set_xlim(0, img_w)
    ax.set_ylim(0, img_h)
    ax.set_aspect('equal')

# Add a single legend for the whole figure
legend_elements = [Line2D([0], [0], marker='*', color='w', label='Other Mouse',
                          markerfacecolor='none', markeredgecolor='b', markersize=15)]
fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
# dashboard = build_general_dashboard(
#     model=value_net,
#     config_counts=config_counts,
#     input_mean=s_mean,
#     input_std=s_std,
#     # No output_mean/std needed for scalar
#     locations=locations,
#     arena_img_array=arena_img,
#     scalar_vmin=robust_vmin,
#     scalar_vmax=robust_vmax,
#     mode='scalar',
#     feature_dim=8
# )
# display(dashboard)

In [ ]:
"""Compare averaged V(s) for easy vs hard configs"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}

# Load image
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h]

# Setup grid
nx, ny = 200, 200
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Pre-calculate mask and config list
valid_mask = get_validity_mask(xx, yy, metadata)
unique_configs = list(config_counts.index)

# Define helper to find config index
def find_config_idx(target_rate):
    for idx, cfg_tuple in enumerate(unique_configs):
        rates = [rate for (_, rate) in cfg_tuple]
        if all(np.isclose(rate, target_rate) for rate in rates):
            return idx, cfg_tuple
    raise ValueError(f"Could not find config with all rates = {target_rate}")

# Get easy and hard config indices
easy_idx, easy_cfg = find_config_idx(0.01)
hard_idx, hard_cfg = find_config_idx(0.002)

# Compute averaged value maps across all other-mouse locations
def compute_value_maps_for_config(cfg_idx):
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    value_maps = []
    value_net.eval()

    for _, (other_x, other_y) in locations.items():

        s_grid = np.zeros((nx * ny, 8), dtype="float32")
        s_grid[:, 0] = xx.ravel()
        s_grid[:, 1] = yy.ravel()
        s_grid[:, 4] = other_x - xx.ravel()
        s_grid[:, 5] = other_y - yy.ravel()

        s_tensor = torch.from_numpy(s_grid).to(device)
        s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)
        s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

        with torch.no_grad():
            vals = value_net(s_full).cpu().numpy()

        v_vals = vals.reshape(ny, nx)
        v_vals_masked = np.where(valid_mask, v_vals, np.nan)
        value_maps.append(v_vals_masked)

    return np.stack(value_maps, axis=0)

all_config_stacks = [compute_value_maps_for_config(cfg_idx) for cfg_idx, _ in enumerate(unique_configs)]
easy_stack = all_config_stacks[easy_idx]
hard_stack = all_config_stacks[hard_idx]

v_easy_mean = np.nanmean(easy_stack, axis=0)
v_hard_mean = np.nanmean(hard_stack, axis=0)
v_diff = v_easy_mean - v_hard_mean

# Plot averaged value difference heatmap
easy_color = "#2a9d8f"
hard_color = "#c46a4a"
neutral_color = "#f6f1e8"
diff_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "easy_hard_diff", [hard_color, neutral_color, easy_color]
)

diff_absmax = np.nanmax(np.abs(v_diff))
diff_absmax = max(diff_absmax, 1e-8)

fig_diff, ax_diff = plt.subplots(figsize=(4.6, 4.2), dpi=300)
ax_diff.set_facecolor(neutral_color)

if arena_img is not None:
    ax_diff.imshow(arena_img, origin="lower", alpha=0.95, extent=extent_img)

im_diff = ax_diff.imshow(
    v_diff,
    origin="lower",
    extent=[0, img_w, 0, img_h],
    cmap=diff_cmap,
    interpolation="bilinear",
    alpha=0.84,
    vmin=-diff_absmax,
    vmax=diff_absmax,
)

cbar = fig_diff.colorbar(im_diff, ax=ax_diff, label="V_easy(s) - V_hard(s)", fraction=0.046, pad=0.03)
cbar.ax.tick_params(labelsize=8)
cbar.outline.set_visible(False)
ax_diff.set_title("Averaged value difference", fontsize=10, fontweight="bold")
ax_diff.set_xlim(0, img_w)
ax_diff.set_ylim(0, img_h)
ax_diff.set_aspect("equal")
ax_diff.set_xticks([])
ax_diff.set_yticks([])
ax_diff.spines[["top", "right", "left", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()

# Plot split violin comparison of averaged values
easy_vals = v_easy_mean[valid_mask]
hard_vals = v_hard_mean[valid_mask]
easy_vals = easy_vals[~np.isnan(easy_vals)]
hard_vals = hard_vals[~np.isnan(hard_vals)]

fig_violin, ax_violin = plt.subplots(figsize=(3.2, 4.2), dpi=300)
ax_violin.set_facecolor("#fcfaf6")
parts_easy = ax_violin.violinplot([easy_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)
parts_hard = ax_violin.violinplot([hard_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)

for body in parts_easy["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.minimum(verts[:, 0], 1)
    body.set_facecolor(easy_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

for body in parts_hard["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.maximum(verts[:, 0], 1)
    body.set_facecolor(hard_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

easy_median = np.nanmedian(easy_vals)
hard_median = np.nanmedian(hard_vals)
ax_violin.scatter([0.9], [easy_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_violin.scatter([1.1], [hard_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_violin.plot([0.78, 1.0], [easy_median, easy_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_violin.plot([1.0, 1.22], [hard_median, hard_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_violin.axvline(1, color="#1f1f1f", lw=0.8, alpha=0.45)
ax_violin.set_xlim(0.5, 1.5)
ax_violin.set_xticks([])
ax_violin.set_ylabel("Average V(s) across locations")
ax_violin.set_title("Averaged V(s) distribution", fontsize=10, fontweight="bold")
ax_violin.text(0.28, 0.98, "Easy", transform=ax_violin.transAxes, ha="center", va="top", color=easy_color, fontsize=9, fontweight="bold")
ax_violin.text(0.72, 0.98, "Hard", transform=ax_violin.transAxes, ha="center", va="top", color=hard_color, fontsize=9, fontweight="bold")
ax_violin.grid(axis="y", alpha=0.16, linewidth=0.8)
ax_violin.spines["left"].set_color("#4a4a4a")
ax_violin.spines["bottom"].set_visible(False)
ax_violin.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# Save all figures as SVG
_value_compare_figs = {
    "heatmap": fig_diff,
    "violin": fig_violin,
}

def _save_value_compare_svg(_):
    _value_compare_figs["heatmap"].savefig("value_diff_easy_minus_hard.svg", bbox_inches="tight")
    _value_compare_figs["violin"].savefig("value_violin_easy_vs_hard.svg", bbox_inches="tight")
    print("Saved: value_diff_easy_minus_hard.svg")
    print("Saved: value_violin_easy_vs_hard.svg")

w_save_value_compare = widgets.Button(description="Save SVG", button_style="success", icon="download")
w_save_value_compare.on_click(_save_value_compare_svg)
display(w_save_value_compare)

In [ ]:
"""Compare averaged V(s) for dominant vs subordinate IQL checkpoints"""

from pathlib import Path

# Set these before running the cell
dominant_ckpt_path = save_dir / "Aeon 3 social 0.2/iql_model_w_c_dominant_1_epoch4.pt"
subordinate_ckpt_path = save_dir / "Aeon 3 social 0.2/iql_model_w_c_1_epoch4.pt"

if not dominant_ckpt_path or not subordinate_ckpt_path:
    raise ValueError("Set both dominant_ckpt_path and subordinate_ckpt_path before running this cell.")

dominant_ckpt_path = Path(dominant_ckpt_path).expanduser()
subordinate_ckpt_path = Path(subordinate_ckpt_path).expanduser()

if not dominant_ckpt_path.exists():
    raise FileNotFoundError(f"Dominant checkpoint not found: {dominant_ckpt_path}")
if not subordinate_ckpt_path.exists():
    raise FileNotFoundError(f"Subordinate checkpoint not found: {subordinate_ckpt_path}")

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y),
}

# Load image
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h]

# Setup grid
nx, ny = 200, 200
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Pre-calculate mask and config list
valid_mask = get_validity_mask(xx, yy, metadata)
unique_configs = list(config_counts.index)

def _load_iql_bundle(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device)

    bundle_value_net = ValueNet(state_dim, hidden_dim).to(device)
    bundle_value_target = ValueNet(state_dim, hidden_dim).to(device)
    bundle_q1_net = QNet(state_dim, action_dim, hidden_dim).to(device)
    bundle_q2_net = QNet(state_dim, action_dim, hidden_dim).to(device)
    bundle_policy_net = GaussianPolicy(state_dim, action_dim, hidden_dim).to(device)

    bundle_value_net.load_state_dict(checkpoint["value_net"])
    bundle_value_target.load_state_dict(checkpoint["value_target"])
    bundle_q1_net.load_state_dict(checkpoint["q1_net"])
    bundle_q2_net.load_state_dict(checkpoint["q2_net"])
    bundle_policy_net.load_state_dict(checkpoint["policy_net"])
    bundle_value_net.eval()

    return {
        "value_net": bundle_value_net,
        "s_mean": checkpoint["s_mean"],
        "s_std": checkpoint["s_std"],
    }

def _compute_value_maps_for_bundle(bundle):
    all_config_stacks = []

    for cfg_idx, _ in enumerate(unique_configs):
        c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
        c_grid[:, cfg_idx] = 1.0
        c_grid_tensor = torch.from_numpy(c_grid).to(device)

        value_maps = []
        for _, (other_x, other_y) in locations.items():
            s_grid = np.zeros((nx * ny, 8), dtype="float32")
            s_grid[:, 0] = xx.ravel()
            s_grid[:, 1] = yy.ravel()
            s_grid[:, 4] = other_x - xx.ravel()
            s_grid[:, 5] = other_y - yy.ravel()

            s_tensor = torch.from_numpy(s_grid).to(device)
            s_norm = (s_tensor - bundle["s_mean"].to(device)) / bundle["s_std"].to(device)
            s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

            with torch.no_grad():
                vals = bundle["value_net"](s_full).cpu().numpy()

            v_vals = vals.reshape(ny, nx)
            value_maps.append(np.where(valid_mask, v_vals, np.nan))

        all_config_stacks.append(np.stack(value_maps, axis=0))

    return np.stack(all_config_stacks, axis=0)

dominant_bundle = _load_iql_bundle(dominant_ckpt_path)
subordinate_bundle = _load_iql_bundle(subordinate_ckpt_path)

dominant_stack = _compute_value_maps_for_bundle(dominant_bundle)
subordinate_stack = _compute_value_maps_for_bundle(subordinate_bundle)

v_dominant_mean = np.nanmean(dominant_stack, axis=(0, 1))
v_subordinate_mean = np.nanmean(subordinate_stack, axis=(0, 1))
v_diff = v_dominant_mean - v_subordinate_mean

dominant_color = "#2a9d8f"
subordinate_color = "#c46a4a"
neutral_color = "#f6f1e8"
diff_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "dominant_subordinate_diff", [subordinate_color, neutral_color, dominant_color]
)

diff_absmax = np.nanmax(np.abs(v_diff))
diff_absmax = max(diff_absmax, 1e-8)

fig_hierarchy_diff, ax_hierarchy_diff = plt.subplots(figsize=(4.6, 4.2), dpi=300)
ax_hierarchy_diff.set_facecolor(neutral_color)

if arena_img is not None:
    ax_hierarchy_diff.imshow(arena_img, origin="lower", alpha=0.95, extent=extent_img)

im_hierarchy_diff = ax_hierarchy_diff.imshow(
    v_diff,
    origin="lower",
    extent=[0, img_w, 0, img_h],
    cmap=diff_cmap,
    interpolation="bilinear",
    alpha=0.84,
    vmin=-diff_absmax,
    vmax=diff_absmax,
)

cbar = fig_hierarchy_diff.colorbar(im_hierarchy_diff, ax=ax_hierarchy_diff, label="V_dominant(s) - V_subordinate(s)", fraction=0.046, pad=0.03)
cbar.ax.tick_params(labelsize=8)
cbar.outline.set_visible(False)
ax_hierarchy_diff.set_title("Averaged value difference by hierarchy", fontsize=10, fontweight="bold")
ax_hierarchy_diff.set_xlim(0, img_w)
ax_hierarchy_diff.set_ylim(0, img_h)
ax_hierarchy_diff.set_aspect("equal")
ax_hierarchy_diff.set_xticks([])
ax_hierarchy_diff.set_yticks([])
ax_hierarchy_diff.spines[["top", "right", "left", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()

dominant_vals = v_dominant_mean[valid_mask]
subordinate_vals = v_subordinate_mean[valid_mask]
dominant_vals = dominant_vals[~np.isnan(dominant_vals)]
subordinate_vals = subordinate_vals[~np.isnan(subordinate_vals)]

fig_hierarchy_violin, ax_hierarchy_violin = plt.subplots(figsize=(3.2, 4.2), dpi=300)
ax_hierarchy_violin.set_facecolor("#fcfaf6")
parts_dominant = ax_hierarchy_violin.violinplot([dominant_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)
parts_subordinate = ax_hierarchy_violin.violinplot([subordinate_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)

for body in parts_dominant["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.minimum(verts[:, 0], 1)
    body.set_facecolor(dominant_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

for body in parts_subordinate["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.maximum(verts[:, 0], 1)
    body.set_facecolor(subordinate_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

dominant_median = np.nanmedian(dominant_vals)
subordinate_median = np.nanmedian(subordinate_vals)
ax_hierarchy_violin.scatter([0.9], [dominant_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_hierarchy_violin.scatter([1.1], [subordinate_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_hierarchy_violin.plot([0.78, 1.0], [dominant_median, dominant_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_hierarchy_violin.plot([1.0, 1.22], [subordinate_median, subordinate_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_hierarchy_violin.axvline(1, color="#1f1f1f", lw=0.8, alpha=0.45)
ax_hierarchy_violin.set_xlim(0.5, 1.5)
ax_hierarchy_violin.set_xticks([])
ax_hierarchy_violin.set_ylabel("Average V(s) across locations")
ax_hierarchy_violin.set_title("Averaged V(s) distribution by hierarchy", fontsize=10, fontweight="bold")
ax_hierarchy_violin.text(0.28, 0.98, "Dominant", transform=ax_hierarchy_violin.transAxes, ha="center", va="top", color=dominant_color, fontsize=9, fontweight="bold")
ax_hierarchy_violin.text(0.72, 0.98, "Subordinate", transform=ax_hierarchy_violin.transAxes, ha="center", va="top", color=subordinate_color, fontsize=9, fontweight="bold")
ax_hierarchy_violin.grid(axis="y", alpha=0.16, linewidth=0.8)
ax_hierarchy_violin.spines["left"].set_color("#4a4a4a")
ax_hierarchy_violin.spines["bottom"].set_visible(False)
ax_hierarchy_violin.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

_hierarchy_compare_figs = {
    "heatmap": fig_hierarchy_diff,
    "violin": fig_hierarchy_violin,
}

def _save_hierarchy_compare_svg(_):
    _hierarchy_compare_figs["heatmap"].savefig("value_diff_dominant_minus_subordinate.svg", bbox_inches="tight")
    _hierarchy_compare_figs["violin"].savefig("value_violin_dominant_vs_subordinate.svg", bbox_inches="tight")
    print("Saved: value_diff_dominant_minus_subordinate.svg")
    print("Saved: value_violin_dominant_vs_subordinate.svg")

w_save_hierarchy_compare = widgets.Button(description="Save SVG", button_style="success", icon="download")
w_save_hierarchy_compare.on_click(_save_hierarchy_compare_svg)
display(w_save_hierarchy_compare)


In [ ]:
"""Compare patch-neighborhood V(s) for patch 1 vs patch 2/3 across all configs"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
other_x, other_y = nest_x, nest_y

# Load image
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]

# Setup grid
nx, ny = 200, 200
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Pre-calculate mask and config list
valid_mask = get_validity_mask(xx, yy, metadata)
unique_configs = list(config_counts.index)

def _patch_extent(*corners):
    xs_patch = np.array([float(c.X) for c in corners], dtype="float32")
    ys_patch = np.array([float(c.Y) for c in corners], dtype="float32")
    return xs_patch.min(), xs_patch.max(), ys_patch.min(), ys_patch.max()

patch_specs = {
    "Patch 1": {
        "center": (patch1_x, patch1_y),
        "extent": _patch_extent(patch1_corner_1, patch1_corner_2, patch1_corner_3, patch1_corner_4),
    },
    "Patch 2": {
        "center": (patch2_x, patch2_y),
        "extent": _patch_extent(patch2_corner_1, patch2_corner_2, patch2_corner_3, patch2_corner_4),
    },
    "Patch 3": {
        "center": (patch3_x, patch3_y),
        "extent": _patch_extent(patch3_corner_1, patch3_corner_2, patch3_corner_3, patch3_corner_4),
    },
}

for patch_name, spec in patch_specs.items():
    x_min, x_max, y_min, y_max = spec["extent"]
    patch_w = x_max - x_min
    patch_h = y_max - y_min
    patch_r = 1.2 * max(patch_w, patch_h)
    patch_dx = xx - spec["center"][0]
    patch_dy = yy - spec["center"][1]
    disk_mask = (patch_dx**2 + patch_dy**2) <= (patch_r**2)
    disk_mask &= valid_mask
    if not np.any(disk_mask):
        raise ValueError(f"No valid grid points found for {patch_name} disk mask")
    spec["radius"] = patch_r
    spec["mask"] = disk_mask

def compute_value_map_for_config_and_other(cfg_idx, other_x, other_y):
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    s_grid = np.zeros((nx * ny, 8), dtype="float32")
    s_grid[:, 0] = xx.ravel()
    s_grid[:, 1] = yy.ravel()
    s_grid[:, 4] = other_x - xx.ravel()
    s_grid[:, 5] = other_y - yy.ravel()

    s_tensor = torch.from_numpy(s_grid).to(device)
    s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)
    s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

    value_net.eval()
    with torch.no_grad():
        vals = value_net(s_full).cpu().numpy()

    v_map = vals.reshape(ny, nx)
    return np.where(valid_mask, v_map, np.nan)

patch1_vals = []
patch23_vals = []

for cfg_idx, _ in enumerate(unique_configs):
    v_map = compute_value_map_for_config_and_other(cfg_idx, other_x, other_y)
    patch1_vals.append(v_map[patch_specs["Patch 1"]["mask"]])
    patch23_vals.append(v_map[patch_specs["Patch 2"]["mask"]])
    patch23_vals.append(v_map[patch_specs["Patch 3"]["mask"]])

patch1_vals = np.concatenate(patch1_vals).astype("float32")
patch23_vals = np.concatenate(patch23_vals).astype("float32")

patch1_vals = patch1_vals[~np.isnan(patch1_vals)]
patch23_vals = patch23_vals[~np.isnan(patch23_vals)]

patch1_color = "#c46a4a"
patch23_color = "#2a9d8f"

fig_patch_violin, ax_patch_violin = plt.subplots(figsize=(3.4, 4.2), dpi=300)
ax_patch_violin.set_facecolor("#fcfaf6")
parts_patch1 = ax_patch_violin.violinplot([patch1_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)
parts_patch23 = ax_patch_violin.violinplot([patch23_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)

for body in parts_patch1["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.minimum(verts[:, 0], 1)
    body.set_facecolor(patch1_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

for body in parts_patch23["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.maximum(verts[:, 0], 1)
    body.set_facecolor(patch23_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

patch1_median = np.nanmedian(patch1_vals)
patch23_median = np.nanmedian(patch23_vals)
ax_patch_violin.scatter([0.9], [patch1_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_patch_violin.scatter([1.1], [patch23_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_patch_violin.plot([0.78, 1.0], [patch1_median, patch1_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_patch_violin.plot([1.0, 1.22], [patch23_median, patch23_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_patch_violin.axvline(1, color="#1f1f1f", lw=0.8, alpha=0.45)
ax_patch_violin.set_xlim(0.5, 1.5)
ax_patch_violin.set_xticks([])
ax_patch_violin.set_ylabel("V(s) in patch-centered disk")
ax_patch_violin.set_title("Patch values with other fixed at nest", fontsize=10, fontweight="bold")
ax_patch_violin.text(0.28, 0.98, "Patch 1", transform=ax_patch_violin.transAxes, ha="center", va="top", color=patch1_color, fontsize=9, fontweight="bold")
ax_patch_violin.text(0.72, 0.98, "Patch 2/3", transform=ax_patch_violin.transAxes, ha="center", va="top", color=patch23_color, fontsize=9, fontweight="bold")
ax_patch_violin.grid(axis="y", alpha=0.16, linewidth=0.8)
ax_patch_violin.spines["left"].set_color("#4a4a4a")
ax_patch_violin.spines["bottom"].set_visible(False)
ax_patch_violin.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

_patch_preference_figs = {
    "violin": fig_patch_violin,
}

def _save_patch_preference_svg(_):
    _patch_preference_figs["violin"].savefig("value_violin_patch23_vs_patch1_all_configs.svg", bbox_inches="tight")
    print("Saved: value_violin_patch23_vs_patch1_all_configs.svg")

w_save_patch_preference = widgets.Button(description="Save SVG", button_style="success", icon="download")
w_save_patch_preference.on_click(_save_patch_preference_svg)
display(w_save_patch_preference)


In [ ]:
"""Compare best-patch V(s) when other is in nest vs at the worst patch"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2

# Load image
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]

# Setup grid
nx, ny = 200, 200
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Pre-calculate mask and config list
valid_mask = get_validity_mask(xx, yy, metadata)
unique_configs = list(config_counts.index)

def _patch_extent(*corners):
    xs_patch = np.array([float(c.X) for c in corners], dtype="float32")
    ys_patch = np.array([float(c.Y) for c in corners], dtype="float32")
    return xs_patch.min(), xs_patch.max(), ys_patch.min(), ys_patch.max()

patch_specs = {
    "Patch1": {
        "center": (patch1_x, patch1_y),
        "extent": _patch_extent(patch1_corner_1, patch1_corner_2, patch1_corner_3, patch1_corner_4),
    },
    "Patch2": {
        "center": (patch2_x, patch2_y),
        "extent": _patch_extent(patch2_corner_1, patch2_corner_2, patch2_corner_3, patch2_corner_4),
    },
    "Patch3": {
        "center": (patch3_x, patch3_y),
        "extent": _patch_extent(patch3_corner_1, patch3_corner_2, patch3_corner_3, patch3_corner_4),
    },
}

for patch_name, spec in patch_specs.items():
    x_min, x_max, y_min, y_max = spec["extent"]
    patch_w = x_max - x_min
    patch_h = y_max - y_min
    patch_r = 1.2 * max(patch_w, patch_h)
    patch_dx = xx - spec["center"][0]
    patch_dy = yy - spec["center"][1]
    disk_mask = (patch_dx**2 + patch_dy**2) <= (patch_r**2)
    disk_mask &= valid_mask
    if not np.any(disk_mask):
        raise ValueError(f"No valid grid points found for {patch_name} disk mask")
    spec["mask"] = disk_mask

def compute_value_map_for_config_and_other(cfg_idx, other_x, other_y):
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    s_grid = np.zeros((nx * ny, 8), dtype="float32")
    s_grid[:, 0] = xx.ravel()
    s_grid[:, 1] = yy.ravel()
    s_grid[:, 4] = other_x - xx.ravel()
    s_grid[:, 5] = other_y - yy.ravel()

    s_tensor = torch.from_numpy(s_grid).to(device)
    s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)
    s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

    value_net.eval()
    with torch.no_grad():
        vals = value_net(s_full).cpu().numpy()

    v_map = vals.reshape(ny, nx)
    return np.where(valid_mask, v_map, np.nan)

nest_vals = []
worst_patch_vals = []

for cfg_idx, cfg_tuple in enumerate(unique_configs):
    cfg_rates = {patch_name: rate for patch_name, rate in cfg_tuple}
    min_rate = min(cfg_rates.values())
    max_rate = max(cfg_rates.values())
    worst_candidates = [patch_name for patch_name, rate in cfg_rates.items() if np.isclose(rate, min_rate)]
    best_candidates = [patch_name for patch_name, rate in cfg_rates.items() if np.isclose(rate, max_rate)]

    if len(worst_candidates) != 1 or len(best_candidates) != 1:
        continue

    best_patch_name = best_candidates[0]
    worst_patch_name = worst_candidates[0]
    worst_other_x, worst_other_y = patch_specs[worst_patch_name]["center"]

    nest_map = compute_value_map_for_config_and_other(cfg_idx, nest_x, nest_y)
    worst_patch_map = compute_value_map_for_config_and_other(cfg_idx, worst_other_x, worst_other_y)

    best_mask = patch_specs[best_patch_name]["mask"]
    nest_vals.append(nest_map[best_mask])
    worst_patch_vals.append(worst_patch_map[best_mask])

nest_vals = np.concatenate(nest_vals).astype("float32")
worst_patch_vals = np.concatenate(worst_patch_vals).astype("float32")

nest_vals = nest_vals[~np.isnan(nest_vals)]
worst_patch_vals = worst_patch_vals[~np.isnan(worst_patch_vals)]

nest_color = "#2a9d8f"
worst_patch_color = "#b05a3c"

fig_social_patch_violin, ax_social_patch_violin = plt.subplots(figsize=(3.5, 4.2), dpi=300)
ax_social_patch_violin.set_facecolor("#fcfaf6")
parts_nest = ax_social_patch_violin.violinplot([nest_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)
parts_worst = ax_social_patch_violin.violinplot([worst_patch_vals], positions=[1], widths=0.9, showmeans=False, showmedians=False, showextrema=False)

for body in parts_nest["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.minimum(verts[:, 0], 1)
    body.set_facecolor(nest_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

for body in parts_worst["bodies"]:
    verts = body.get_paths()[0].vertices
    verts[:, 0] = np.maximum(verts[:, 0], 1)
    body.set_facecolor(worst_patch_color)
    body.set_edgecolor("#1f1f1f")
    body.set_alpha(0.85)

nest_median = np.nanmedian(nest_vals)
worst_patch_median = np.nanmedian(worst_patch_vals)
ax_social_patch_violin.scatter([0.9], [nest_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_social_patch_violin.scatter([1.1], [worst_patch_median], color="white", edgecolor="#1f1f1f", s=38, zorder=4)
ax_social_patch_violin.plot([0.78, 1.0], [nest_median, nest_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_social_patch_violin.plot([1.0, 1.22], [worst_patch_median, worst_patch_median], color="#1f1f1f", lw=1.6, zorder=3)
ax_social_patch_violin.axvline(1, color="#1f1f1f", lw=0.8, alpha=0.45)
ax_social_patch_violin.set_xlim(0.5, 1.5)
ax_social_patch_violin.set_xticks([])
ax_social_patch_violin.set_ylabel("V(s) in best-patch disk")
ax_social_patch_violin.set_title("Best patch value: nest vs worst patch", fontsize=10, fontweight="bold", pad=26)
ax_social_patch_violin.text(0.28, 1.02, "Other in nest", transform=ax_social_patch_violin.transAxes, ha="center", va="bottom", color=nest_color, fontsize=8.5, fontweight="bold", clip_on=False)
ax_social_patch_violin.text(0.72, 1.02, "Other at\nworst patch", transform=ax_social_patch_violin.transAxes, ha="center", va="bottom", color=worst_patch_color, fontsize=8.5, fontweight="bold", clip_on=False, linespacing=0.9)
ax_social_patch_violin.grid(axis="y", alpha=0.16, linewidth=0.8)
ax_social_patch_violin.spines["left"].set_color("#4a4a4a")
ax_social_patch_violin.spines["bottom"].set_visible(False)
ax_social_patch_violin.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

_social_patch_compare_figs = {
    "violin": fig_social_patch_violin,
}

def _save_social_patch_compare_svg(_):
    _social_patch_compare_figs["violin"].savefig("value_violin_best_patch_nest_vs_worst_patch.svg", bbox_inches="tight")
    print("Saved: value_violin_best_patch_nest_vs_worst_patch.svg")

w_save_social_patch_compare = widgets.Button(description="Save SVG", button_style="success", icon="download")
w_save_social_patch_compare.on_click(_save_social_patch_compare_svg)
display(w_save_social_patch_compare)


In [ ]:
"""Plot navigation policy for different fixed other positions"""

# Define locations
nest_x = (float(nest_corner_1.X) + float(nest_corner_3.X)) / 2
nest_y = (float(nest_corner_1.Y) + float(nest_corner_3.Y)) / 2
patch1_x = (float(patch1_corner_1.X) + float(patch1_corner_3.X)) / 2
patch1_y = (float(patch1_corner_1.Y) + float(patch1_corner_3.Y)) / 2
patch2_x = (float(patch2_corner_1.X) + float(patch2_corner_3.X)) / 2
patch2_y = (float(patch2_corner_1.Y) + float(patch2_corner_3.Y)) / 2
patch3_x = (float(patch3_corner_1.X) + float(patch3_corner_3.X)) / 2
patch3_y = (float(patch3_corner_1.Y) + float(patch3_corner_3.Y)) / 2
corridor_r = (inner_radius + outer_radius) / 2

locations = {
    "Arena Center": (center_x, center_y),
    "Nest Area": (nest_x, nest_y),
    "Gate": (center_x - corridor_r, center_y),
    "Patch 1": (patch1_x, patch1_y),
    "Patch 2": (patch2_x, patch2_y),
    "Patch 3": (patch3_x, patch3_y)
}
n_locs = len(locations)

# Load image and determine global bounds
arena_img = plt.imread("arena.png")
img_h, img_w = arena_img.shape[:2]
extent_img = [0, img_w, 0, img_h] # Image uses its own full coords

# Setup grid (Full Image)
nx, ny = 30, 30
xs = np.linspace(0, img_w, nx)
ys = np.linspace(0, img_h, ny)
xx, yy = np.meshgrid(xs, ys)

# Prepare figure
fig, axes = plt.subplots(
    n_configs, n_locs, 
    figsize=(4 * n_locs, 4 * n_configs), 
    squeeze=False,
    dpi=300
)

# Loop over configs (rows) and locations (columns)
unique_configs = list(config_counts.index)
for cfg_idx, cfg_tuple in enumerate(unique_configs):
    rates = [f"{rate:.4f}" for (_, rate) in cfg_tuple]
    # Format: (Rate1, Rate2, Rate3)
    cfg_name = f"({rates[0]}, {rates[1]}, {rates[2]})"
    
    # One-hot encoding for config
    c_grid = np.zeros((nx * ny, n_configs), dtype="float32")
    c_grid[:, cfg_idx] = 1.0
    c_grid_tensor = torch.from_numpy(c_grid).to(device)

    for loc_j, (loc_name, (other_x, other_y)) in enumerate(locations.items()):
        ax = axes[cfg_idx, loc_j]
        
        print(f"Computing policy for config {cfg_name} with fixed other at {loc_name}...")

        # Construct state
        s_grid = np.zeros((nx * ny, 8), dtype="float32")
        s_grid[:, 0] = xx.ravel() # x_self
        s_grid[:, 1] = yy.ravel() # y_self
        s_grid[:, 4] = other_x - xx.ravel() # dx
        s_grid[:, 5] = other_y - yy.ravel() # dy
        
        # Normalize
        s_tensor = torch.from_numpy(s_grid).to(device)
        s_norm = (s_tensor - s_mean.to(device)) / s_std.to(device)

        s_full = torch.cat([s_norm, c_grid_tensor], dim=1)

        # Query IQL policy
        policy_net.eval()
        with torch.no_grad():
            mu, _ = policy_net(s_full)
            mu = mu.cpu().numpy()

        # Un-normalize actions
        a_std_np = a_std.detach().cpu().numpy()
        a_mean_np = a_mean.detach().cpu().numpy()
        actions = (mu * a_std_np) + a_mean_np

        u = actions[:, 0].reshape(ny, nx)
        v = actions[:, 1].reshape(ny, nx)

        # Apply masking
        valid_start = get_validity_mask(xx, yy, metadata)
        u = np.where(valid_start, u, np.nan)
        v = np.where(valid_start, v, np.nan)

        # Plot background (use full image extent)
        if arena_img is not None:
            ax.imshow(arena_img, origin="lower", alpha=0.5, extent=extent_img)

        # Plot self movement (arrows)
        ax.quiver(xx, yy, u, v, color='red', scale=None, scale_units='inches')

        # Plot fixed other mouse (star)
        ax.plot(other_x, other_y, 'b*', markersize=25, markeredgecolor='white', label=f"Other ({loc_name})")

        # Labeling Logic:
        # Column Labels (Top Row only)
        if cfg_idx == 0:
            ax.set_title(loc_name, fontsize=14, fontweight='bold')
        else:
            ax.set_title("") 

        # Row Labels (Left Column only)
        if loc_j == 0:
            # Rotated 90 degrees
            ax.set_ylabel(f"Config\n{cfg_name}", fontsize=12, fontweight='bold', rotation=90, labelpad=20)
            # Remove standard y ticks to clean up
            ax.set_yticks([]) 
        else:
            ax.set_ylabel("")
            ax.set_yticks([])

        # Remove x ticks for all plots to clean up
        ax.set_xticks([])
        
        # Lock view to full arena dimensions
        ax.set_xlim(0, img_w)
        ax.set_ylim(0, img_h)
        
        ax.set_aspect('equal')

# Add a single legend for the whole figure
legend_elements = [Line2D([0], [0], marker='*', color='w', label='Other Mouse',
                          markerfacecolor='b', markersize=15)]
fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(0.0, 1.0))

plt.tight_layout()
plt.show()

In [ ]:
# dashboard = build_general_dashboard(
#     model=policy_net,
#     config_counts=config_counts,
#     input_mean=s_mean,
#     input_std=s_std,
#     output_mean=a_mean,
#     output_std=a_std,
#     locations=locations,
#     arena_img_array=arena_img,
#     mode='vector',
#     feature_dim=8
# )
# display(dashboard)